# Notebook 01 -- Adquisición de Datos: Demanda Eléctrica Horaria del SIN de Colombia

---


## Objetivo

Descargar y validar 5 años de datos horarios de demanda eléctrica del SIN desde la API pública de SINERGOX (XM). Incluye análisis exploratorio completo: patrones temporales, estacionariedad y EDA.


## Entrada

API SINERGOX (XM S.A.) — Demanda Real del SIN horaria (2020–2025)


## Salida

data/raw/demand.parquet (52,608 registros) + data/raw/metadata.json


## Instalación

In [ ]:
# Instalación de dependencias
# En Docker (make up / make pipeline) ya vienen instaladas con versiones fijas
# (requirements.txt). En Google Colab, después de clonar el repo y hacer
# %cd notebooks, se instalan las mismas versiones:
import sys
if "google.colab" in sys.modules:
    !pip install -q -r ../requirements.txt

## Configuración

In [ ]:
# ============================================================
# CONFIGURACIÓN — Modificar según entorno
# ============================================================
DATA_DIR = "../data"
MODELS_DIR = "../models"
RESULTS_DIR = "../results"
# Figuras en inglés (versión para publicación); la carpeta no se versiona
os.makedirs(f"{RESULTS_DIR}/english", exist_ok=True)
RANDOM_SEED = 42
N_JOBS = -1

import os
os.makedirs(f"{DATA_DIR}/raw", exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

## Imports

In [ ]:
import json
import time
import warnings
from datetime import datetime, timedelta
from typing import List, Tuple, Optional, Dict, Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mticker
import seaborn as sns
import holidays
from statsmodels.tsa.seasonal import STL
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

import requests

warnings.filterwarnings("ignore")

# Estilo de gráficos
sns.set_style("whitegrid")
plt.rcParams.update({
    "figure.figsize": (14, 5),
    "figure.dpi": 120,
    "axes.titlesize": 14,
    "axes.labelsize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "font.family": "serif",
})

print(f"pandas: {pd.__version__}")
print(f"numpy: {np.__version__}")

## 1. Funciones de ingestión de datos (inline)

La API pública de SINERGOX (XM) permite consultar datos horarios del SIN mediante POST a:

```
https://servapibi.xm.com.co/hourly
```

**Restricciones de la API:**
- Máximo **30 días** por consulta.
- Sin autenticación requerida.
- La variable `DemandaReal` contiene la demanda total del SIN en kWh por hora.

Las funciones definidas a continuación implementan:
1. Iteración por ventanas de 30 días para cubrir N años.
2. Reintentos con backoff exponencial (3 intentos).
3. Validación de completitud (24 registros por día).
4. Validación de rango (4,000–12,000 kWh para el SIN histórico).
5. Eliminación de timestamps duplicados.

In [ ]:
# ============================================================
# Funciones de ingestión — API XM (servapibi.xm.com.co)
# Ventanas y payload alineados con el cliente JS de referencia (ReadDB)
# ============================================================
API_URL = "https://servapibi.xm.com.co/hourly"
DEMAND_VARIABLE = "DemaReal"
MAX_RETRIES = 3
INITIAL_BACKOFF = 2  # segundos

# Rango histórico válido de demanda horaria del SIN (kWh): ~4.000-12.000 MWh
DEMAND_MIN_KWH = 4_000_000
DEMAND_MAX_KWH = 12_000_000


def generate_date_windows(
    start_date: str, end_date: str
) -> List[Tuple[str, str]]:
    """
    Genera ventanas de fechas para iterar sobre la API de XM.

    Cada ventana cubre UN SOLO DÍA (24 registros por request), igual que el
    cliente JS de referencia para datos horarios (HourlyEntities). Ventanas
    más largas (p. ej. 30 días = 720 registros) probablemente exceden el
    límite interno de la API y producen respuestas vacías o truncadas.

    Parameters
    ----------
    start_date : str  — Fecha inicio 'YYYY-MM-DD'.
    end_date : str    — Fecha fin 'YYYY-MM-DD'.

    Returns
    -------
    List[Tuple[str, str]]
        Lista de tuplas (inicio, fin) donde inicio == fin (un día por ventana).
    """
    windows = []
    current = datetime.strptime(start_date, "%Y-%m-%d")
    end = datetime.strptime(end_date, "%Y-%m-%d")

    while current <= end:
        # Ventana de 1 solo día: start == end (alineado con JS)
        date_str = current.strftime("%Y-%m-%d")
        windows.append((date_str, date_str))
        current += timedelta(days=1)

    return windows


def _parse_hourly_items(items: list) -> list:
    """
    Parsea la respuesta de la API de XM, soportando AMBOS formatos
    posibles de HourlyEntities:

    Formato A (lista de {Hour, Value}):
        "HourlyEntities": [
            {"Hour": 1, "Value": 5432.1},
            {"Hour": 2, "Value": 5210.3}, ...
        ]

    Formato B (lista de entidades con Values dict):
        "HourlyEntities": [
            {
                "Id": "Sistema",
                "Values": {"Hour01": 5432.1, "Hour02": 5210.3, ...}
            }
        ]

    Este parsing defensivo evita fallos silenciosos si la estructura
    de la respuesta no coincide con lo esperado.
    """
    records = []

    for item in items:
        date_str = item.get("Date", "")
        if not date_str:
            continue

        base_date = pd.to_datetime(date_str).normalize()
        hourly_entities = item.get("HourlyEntities", [])

        for entity in hourly_entities:
            # --- Formato A: cada entity tiene Hour y Value directamente ---
            if "Hour" in entity and "Value" in entity:
                hour = entity["Hour"]  # Horas 1-24 en la API
                value = entity.get("Value")
                if value is not None:
                    dt = base_date + pd.Timedelta(hours=int(hour) - 1)
                    records.append({"datetime": dt, "demand_kwh": float(value)})

            # --- Formato B: entity tiene Id + Values dict ---
            elif "Values" in entity:
                values_dict = entity["Values"]
                for key, value in values_dict.items():
                    # Claves tipo "Hour01", "Hour02", ..., "Hour24"
                    if key.startswith("Hour") and value is not None:
                        try:
                            hour = int(key.replace("Hour", ""))
                            dt = base_date + pd.Timedelta(hours=hour - 1)
                            records.append({"datetime": dt, "demand_kwh": float(value)})
                        except (ValueError, TypeError):
                            continue

            # --- Formato desconocido: logear para debug ---
            else:
                print(f"  [WARN] Estructura inesperada en HourlyEntities: {list(entity.keys())}")

    return records


def fetch_xm_demand(
    start_date: str,
    end_date: str,
    max_retries: int = MAX_RETRIES,
    initial_backoff: float = INITIAL_BACKOFF,
) -> Optional[pd.DataFrame]:
    """
    Descarga datos horarios de demanda real del SIN desde la API de XM
    para una ventana de fechas (idealmente 1 día).

    Notas:
    1. El payload incluye "Filter": [] (como el cliente JS): algunas APIs
       validan la presencia de todos los campos del schema aunque estén vacíos.
    2. Parsing defensivo con soporte para múltiples formatos de respuesta.

    Parameters
    ----------
    start_date : str  — Fecha inicio 'YYYY-MM-DD'.
    end_date : str    — Fecha fin 'YYYY-MM-DD'.
    max_retries : int — Número máximo de reintentos.
    initial_backoff : float — Backoff inicial en segundos.

    Returns
    -------
    pd.DataFrame or None
    """
    payload = {
        "MetricId": DEMAND_VARIABLE,
        "StartDate": start_date,
        "EndDate": end_date,
        "Entity": "Sistema",
        "Filter": [],  # requerido aunque vaya vacío
    }
    headers = {"Content-Type": "application/json"}

    for attempt in range(1, max_retries + 1):
        try:
            response = requests.post(
                API_URL, json=payload, headers=headers, timeout=60
            )
            response.raise_for_status()
            data = response.json()

            # Parsing defensivo que soporta ambos formatos de respuesta
            items = data.get("Items", [])
            records = _parse_hourly_items(items)

            if records:
                return pd.DataFrame(records)
            else:
                print(f"  [WARN] Sin registros para {start_date} -> {end_date}")
                return None

        except requests.exceptions.RequestException as e:
            wait = initial_backoff * (2 ** (attempt - 1))
            print(
                f"  [RETRY {attempt}/{max_retries}] Error: {e}. "
                f"Reintentando en {wait:.0f}s..."
            )
            time.sleep(wait)

    print(f"  [ERROR] Fallaron todos los intentos para {start_date} -> {end_date}")
    return None


def download_demand_data(
    start_date: str,
    end_date: str,
) -> pd.DataFrame:
    """
    Descarga datos de demanda horaria del SIN para un rango de fechas
    arbitrario, iterando día a día (alineado con el cliente JS).

    Nota sobre eficiencia: con ventanas de 1 día, un año requiere ~365
    requests. Con el sleep de 0.3s por request, eso toma ~2 min.
    Es el tradeoff correcto: la API responde consistentemente vs. ventanas
    grandes que devuelven vacío.

    Parameters
    ----------
    start_date : str  — Fecha inicio 'YYYY-MM-DD'.
    end_date : str    — Fecha fin 'YYYY-MM-DD'.

    Returns
    -------
    pd.DataFrame
        DataFrame consolidado ['datetime', 'demand_kwh'], ordenado, sin duplicados.
    """
    windows = generate_date_windows(start_date, end_date)
    print(f"Total de ventanas (días) a consultar: {len(windows)}")
    print(f"Período: {start_date} -> {end_date}")
    print("=" * 60)

    all_dfs = []
    failed_days = []

    for i, (ws, we) in enumerate(windows, 1):
        df = fetch_xm_demand(ws, we)
        if df is not None:
            all_dfs.append(df)
        else:
            failed_days.append(ws)
            print("SIN DATOS")

        # Pausa entre requests para no saturar la API
        # (el JS usa Promise.all sin throttling, pero ser conservador es mejor)
        time.sleep(0.3)

    if not all_dfs:
        raise RuntimeError("No se obtuvieron datos de la API.")

    # Consolidar
    df_full = pd.concat(all_dfs, ignore_index=True)

    # Eliminar duplicados por timestamp
    n_before = len(df_full)
    df_full = df_full.drop_duplicates(subset="datetime", keep="first")
    n_dupes = n_before - len(df_full)
    if n_dupes > 0:
        print(f"\n[INFO] Se eliminaron {n_dupes} timestamps duplicados.")

    # Ordenar
    df_full = df_full.sort_values("datetime").reset_index(drop=True)

    print(f"\n{'=' * 60}")
    print(f"Descarga completada: {len(df_full):,} registros")
    print(f"Rango: {df_full['datetime'].min()} -> {df_full['datetime'].max()}")

    if failed_days:
        print(f"[WARN] Días sin datos ({len(failed_days)}): {failed_days[:10]}...")

    return df_full


def validate_demand_data(df: pd.DataFrame) -> Dict[str, Any]:
    """
    Valida completitud y calidad de los datos de demanda descargados.

    Checks:
    1. 24 registros por día (detecta días incompletos).
    2. Rango válido: 4,000 - 12,000 kWh.
    3. Detecta gaps temporales (horas faltantes).
    4. Valores nulos.

    Parameters
    ----------
    df : pd.DataFrame  — DataFrame con columnas ['datetime', 'demand_kwh'].

    Returns
    -------
    Dict[str, Any]  — Diccionario con resultados de validación.
    """
    report = {}

    # --- Valores nulos ---
    n_nulls = df["demand_kwh"].isna().sum()
    report["n_nulls"] = int(n_nulls)
    print(f"Valores nulos en demand_kwh: {n_nulls}")

    # --- Registros por día ---
    df_temp = df.copy()
    df_temp["date"] = df_temp["datetime"].dt.date
    records_per_day = df_temp.groupby("date").size()
    incomplete_days = records_per_day[records_per_day != 24]
    report["n_incomplete_days"] = len(incomplete_days)
    if len(incomplete_days) > 0:
        print(f"Días con != 24 registros: {len(incomplete_days)}")
        print(incomplete_days.head(10))
    else:
        print("Todos los días tienen exactamente 24 registros.")

    # --- Rango de valores ---
    out_of_range = df[
        (df["demand_kwh"] < DEMAND_MIN_KWH) | (df["demand_kwh"] > DEMAND_MAX_KWH)
    ]
    report["n_out_of_range"] = len(out_of_range)
    if len(out_of_range) > 0:
        print(f"\nRegistros fuera de rango [{DEMAND_MIN_KWH}, {DEMAND_MAX_KWH}] kWh: {len(out_of_range)}")
        print(f"  Mín observado: {df['demand_kwh'].min():.1f} kWh")
        print(f"  Máx observado: {df['demand_kwh'].max():.1f} kWh")
    else:
        print(f"Todos los valores en rango [{DEMAND_MIN_KWH}, {DEMAND_MAX_KWH}] kWh.")

    # --- Gaps temporales ---
    expected_range = pd.date_range(
        start=df["datetime"].min(),
        end=df["datetime"].max(),
        freq="h",
    )
    actual_set = set(df["datetime"])
    missing_hours = sorted(set(expected_range) - actual_set)
    report["n_missing_hours"] = len(missing_hours)
    report["missing_hours"] = [str(h) for h in missing_hours[:50]]

    if missing_hours:
        print(f"\nHoras faltantes (gaps): {len(missing_hours)}")
        gap_blocks = []
        block_start = missing_hours[0]
        prev = missing_hours[0]
        for h in missing_hours[1:]:
            if h - prev > timedelta(hours=1):
                gap_blocks.append((block_start, prev))
                block_start = h
            prev = h
        gap_blocks.append((block_start, prev))
        report["gap_blocks"] = [
            {"start": str(s), "end": str(e)} for s, e in gap_blocks
        ]
        print(f"Bloques de gaps contiguos: {len(gap_blocks)}")
        for s, e in gap_blocks[:10]:
            duration = (e - s).total_seconds() / 3600 + 1
            print(f"  {s} -> {e}  ({duration:.0f}h)")
    else:
        print("Sin gaps temporales detectados.")
        report["gap_blocks"] = []

    # --- Estadísticas básicas ---
    report["stats"] = {
        "mean": float(df["demand_kwh"].mean()),
        "std": float(df["demand_kwh"].std()),
        "min": float(df["demand_kwh"].min()),
        "max": float(df["demand_kwh"].max()),
        "median": float(df["demand_kwh"].median()),
    }
    print(f"\nEstadísticas básicas:")
    print(f"  Media:   {report['stats']['mean']:.1f} kWh")
    print(f"  Std:     {report['stats']['std']:.1f} kWh")
    print(f"  Mín:     {report['stats']['min']:.1f} kWh")
    print(f"  Máx:     {report['stats']['max']:.1f} kWh")
    print(f"  Mediana: {report['stats']['median']:.1f} kWh")

    return report

## 2. Ejecutar ingestión

Se descargan 5 años completos de datos horarios (2021–2025). La descarga tomará varios minutos debido a las ~1800 ventanas de 1 días y la pausa entre consultas.

In [ ]:
# Descargar datos
START_DATE = "2020-01-01"
END_DATE = "2025-12-31"

# La descarga hace ~2,200 peticiones (una por día) y tarda ~45 min. Si ya existe
# data/raw/demand.parquet se reutiliza; para volver a descargar: FORCE_DOWNLOAD=1
raw_path = f"{DATA_DIR}/raw/demand.parquet"
meta_path = f"{DATA_DIR}/raw/metadata.json"
if os.path.exists(raw_path) and os.environ.get("FORCE_DOWNLOAD") != "1":
    df = pd.read_parquet(raw_path)
    with open(meta_path) as f:
        DOWNLOAD_TIMESTAMP = json.load(f)["download_timestamp"]
    print(f"Usando datos ya descargados: {raw_path} ({len(df):,} registros, descargados el {DOWNLOAD_TIMESTAMP})")
else:
    df = download_demand_data(START_DATE, END_DATE)
    DOWNLOAD_TIMESTAMP = datetime.now().isoformat()

In [ ]:
# Validar
print("\n" + "=" * 60)
print("VALIDACIÓN DE DATOS")
print("=" * 60)
validation_report = validate_demand_data(df)

print(f"\nShape final: {df.shape}")
df.head(10)

---

## 3. Análisis Exploratorio de Datos (EDA)

A continuación se realiza un análisis exploratorio completo de la serie temporal de demanda eléctrica. El objetivo es entender los patrones temporales (diarios, semanales, anuales), detectar anomalías, y fundamentar las decisiones de modelado.

### 3.1 Serie temporal completa

In [ ]:
# --- 6.1 Serie temporal completa ---
fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(df["datetime"], df["demand_kwh"], linewidth=0.3, alpha=0.8, color="#2c3e50")

# Media móvil semanal para tendencia
df_plot = df.set_index("datetime").copy()
rolling_mean = df_plot["demand_kwh"].rolling(window=168, center=True).mean()
ax.plot(rolling_mean.index, rolling_mean.values, linewidth=1.5, color="#e74c3c",
        label="Media móvil 7 días")

ax.set_title("Demanda Energía Eléctrica Horaria", fontweight="bold")
ax.set_xlabel("Fecha")
ax.set_ylabel("Demanda (kWh)")
ax.legend(loc="upper right")
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_minor_locator(mdates.MonthLocator(bymonth=[4, 7, 10]))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/01_timeseries_completa.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(df["datetime"], df["demand_kwh"], linewidth=0.3, alpha=0.8, color="#2c3e50")

df_plot = df.set_index("datetime").copy()
rolling_mean = df_plot["demand_kwh"].rolling(window=168, center=True).mean()
ax.plot(
    rolling_mean.index,
    rolling_mean.values,
    linewidth=1.5,
    color="#e74c3c",
    label="7-day moving average"
)

ax.set_title("Hourly Electricity Demand of the Colombian", fontweight="bold")
ax.set_xlabel("Date")
ax.set_ylabel("Demand (kWh)")
ax.legend(loc="upper right")
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_minor_locator(mdates.MonthLocator(bymonth=[4, 7, 10]))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/english/01_timeseries_completa.png", dpi=150, bbox_inches="tight")
plt.show()

### 3.2 Distribución de demanda por hora del día

El perfil horario de carga eléctrica del SIN refleja el ciclo de actividad económica y residencial: demanda mínima en la madrugada (3–4 AM), rampa matutina, pico de mediodía, valle de siesta, y pico vespertino (19–20h).

In [ ]:
# --- 6.2 Distribución por hora del día ---
df["hour"] = df["datetime"].dt.hour

fig, ax = plt.subplots(figsize=(14, 6))
bp = ax.boxplot(
    [df[df["hour"] == h]["demand_kwh"].dropna().values for h in range(24)],
    positions=range(24),
    widths=0.6,
    patch_artist=True,
    showfliers=True,
    flierprops={"marker": ".", "markersize": 2, "alpha": 0.3},
    medianprops={"color": "#c0392b", "linewidth": 1.5},
)

# Colorear boxes con gradiente
colors = plt.cm.YlOrRd(np.linspace(0.15, 0.85, 24))
for patch, color in zip(bp["boxes"], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.set_title("Distribución de Demanda por Hora del Día", fontweight="bold")
ax.set_xlabel("Hora del día")
ax.set_ylabel("Demanda (kWh)")
ax.set_xticks(range(24))
ax.set_xticklabels([f"{h:02d}" for h in range(24)])
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/01_dist_por_hora.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
df["hour"] = df["datetime"].dt.hour

fig, ax = plt.subplots(figsize=(14, 6))
bp = ax.boxplot(
    [df[df["hour"] == h]["demand_kwh"].dropna().values for h in range(24)],
    positions=range(24),
    widths=0.6,
    patch_artist=True,
    showfliers=True,
    flierprops={"marker": ".", "markersize": 2, "alpha": 0.3},
    medianprops={"color": "#c0392b", "linewidth": 1.5},
)

# Color boxes with a gradient
colors = plt.cm.YlOrRd(np.linspace(0.15, 0.85, 24))
for patch, color in zip(bp["boxes"], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.set_title("Demand Distribution by Hour of the Day", fontweight="bold")
ax.set_xlabel("Hour of the Day")
ax.set_ylabel("Demand (kWh)")
ax.set_xticks(range(24))
ax.set_xticklabels([f"{h:02d}" for h in range(24)])
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/english/01_dist_by_hour.png", dpi=150, bbox_inches="tight")
plt.show()

### 3.3 Distribución por día de la semana

La demanda presenta un patrón semanal marcado: días laborales (lunes a viernes) con mayor demanda que fines de semana (sábado y domingo). Los domingos y festivos muestran los niveles más bajos.

In [ ]:
# --- 6.3 Distribución por día de la semana ---
df["day_of_week"] = df["datetime"].dt.dayofweek  # 0=Lunes, 6=Domingo
day_labels = ["Lun", "Mar", "Mié", "Jue", "Vie", "Sáb", "Dom"]

fig, ax = plt.subplots(figsize=(10, 6))
bp = ax.boxplot(
    [df[df["day_of_week"] == d]["demand_kwh"].dropna().values for d in range(7)],
    positions=range(7),
    widths=0.6,
    patch_artist=True,
    showfliers=True,
    flierprops={"marker": ".", "markersize": 2, "alpha": 0.3},
    medianprops={"color": "#c0392b", "linewidth": 1.5},
)

weekday_colors = ["#3498db"] * 5 + ["#e67e22"] * 2
for patch, color in zip(bp["boxes"], weekday_colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.set_title("Distribución de Demanda por Día de la Semana", fontweight="bold")
ax.set_xlabel("Día de la semana")
ax.set_ylabel("Demanda (kWh)")
ax.set_xticks(range(7))
ax.set_xticklabels(day_labels)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/01_dist_por_dia_semana.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
df["day_of_week"] = df["datetime"].dt.dayofweek
day_labels = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]

fig, ax = plt.subplots(figsize=(10, 6))
bp = ax.boxplot(
    [df[df["day_of_week"] == d]["demand_kwh"].dropna().values for d in range(7)],
    positions=range(7),
    widths=0.6,
    patch_artist=True,
    showfliers=True,
    flierprops={"marker": ".", "markersize": 2, "alpha": 0.3},
    medianprops={"color": "#c0392b", "linewidth": 1.5},
)

weekday_colors = ["#3498db"] * 5 + ["#e67e22"] * 2
for patch, color in zip(bp["boxes"], weekday_colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.set_title("Demand Distribution by Day of the Week", fontweight="bold")
ax.set_xlabel("Day of the Week")
ax.set_ylabel("Demand (kWh)")
ax.set_xticks(range(7))
ax.set_xticklabels(day_labels)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/english/01_dist_by_day_of_week.png", dpi=150, bbox_inches="tight")
plt.show()

### 3.4 Distribución por mes

Colombia, al estar en la zona ecuatorial, no presenta estaciones térmicas marcadas. Sin embargo, la demanda eléctrica varía a lo largo del año por factores como: temporadas de vacaciones (diciembre–enero, Semana Santa), actividad industrial, y eventos climáticos (fenómenos de El Niño/La Niña que afectan la generación hidroeléctrica y, indirectamente, la demanda).

In [ ]:
# --- 6.4 Distribución por mes ---
df["month"] = df["datetime"].dt.month
month_labels = ["Ene", "Feb", "Mar", "Abr", "May", "Jun",
                "Jul", "Ago", "Sep", "Oct", "Nov", "Dic"]

fig, ax = plt.subplots(figsize=(14, 6))
bp = ax.boxplot(
    [df[df["month"] == m]["demand_kwh"].dropna().values for m in range(1, 13)],
    positions=range(12),
    widths=0.6,
    patch_artist=True,
    showfliers=True,
    flierprops={"marker": ".", "markersize": 2, "alpha": 0.3},
    medianprops={"color": "#c0392b", "linewidth": 1.5},
)

month_colors = plt.cm.viridis(np.linspace(0.1, 0.9, 12))
for patch, color in zip(bp["boxes"], month_colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.set_title("Distribución de Demanda por Mes", fontweight="bold")
ax.set_xlabel("Mes")
ax.set_ylabel("Demanda (kWh)")
ax.set_xticks(range(12))
ax.set_xticklabels(month_labels)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/01_dist_por_mes.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
df["month"] = df["datetime"].dt.month
month_labels = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
                "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

fig, ax = plt.subplots(figsize=(14, 6))
bp = ax.boxplot(
    [df[df["month"] == m]["demand_kwh"].dropna().values for m in range(1, 13)],
    positions=range(12),
    widths=0.6,
    patch_artist=True,
    showfliers=True,
    flierprops={"marker": ".", "markersize": 2, "alpha": 0.3},
    medianprops={"color": "#c0392b", "linewidth": 1.5},
)

month_colors = plt.cm.viridis(np.linspace(0.1, 0.9, 12))
for patch, color in zip(bp["boxes"], month_colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.set_title("Demand Distribution by Month", fontweight="bold")
ax.set_xlabel("Month")
ax.set_ylabel("Demand (kWh)")
ax.set_xticks(range(12))
ax.set_xticklabels(month_labels)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/english/01_dist_by_month.png", dpi=150, bbox_inches="tight")
plt.show()

### 3.5 Heatmap: Hora x Día de la semana

Este heatmap muestra la demanda promedio para cada combinación de hora del día y día de la semana. Permite visualizar simultáneamente los patrones diarios y semanales, e identificar las horas y días de mayor y menor demanda.

In [ ]:
# --- 6.5 Heatmap hora x día de la semana ---
pivot = df.pivot_table(
    values="demand_kwh",
    index="hour",
    columns="day_of_week",
    aggfunc="mean",
)
pivot.columns = day_labels

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    pivot,
    cmap="YlOrRd",
    annot=True,
    fmt=".0f",
    linewidths=0.5,
    linecolor="white",
    cbar_kws={"label": "Demanda promedio (kWh)"},
    ax=ax,
)
ax.set_title("Demanda Promedio (kWh): Hora del Día vs Día de la Semana", fontweight="bold")
ax.set_xlabel("Día de la semana")
ax.set_ylabel("Hora del día")
ax.set_yticklabels([f"{h:02d}:00" for h in range(24)], rotation=0)
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/01_heatmap_hora_dia.png", dpi=150, bbox_inches="tight")
plt.show()

### 3.6 Autocorrelación (ACF / PACF)

Se analizan las funciones de autocorrelación hasta lag 336 (14 días = 2 semanas) para identificar:
- **Lag 24:** Periodicidad diaria (patrón horario que se repite cada día).
- **Lag 168:** Periodicidad semanal (patrón que se repite cada semana).
- **Lag 336:** Dos semanas, para verificar persistencia del ciclo semanal.

In [ ]:
# --- 6.6 ACF / PACF ---
series = df.set_index("datetime")["demand_kwh"].dropna()

fig, axes = plt.subplots(2, 1, figsize=(16, 8))

plot_acf(series, lags=336, ax=axes[0], alpha=0.05,
         title="Autocorrelación (ACF) — Demanda Horaria")
axes[0].axvline(x=24, color="#e74c3c", linestyle="--", alpha=0.7, label="Lag 24 (diario)")
axes[0].axvline(x=168, color="#2ecc71", linestyle="--", alpha=0.7, label="Lag 168 (semanal)")
axes[0].legend(loc="upper right")

plot_pacf(series, lags=336, ax=axes[1], alpha=0.05, method="ywm",
          title="Autocorrelación Parcial (PACF) — Demanda Horaria")
axes[1].axvline(x=24, color="#e74c3c", linestyle="--", alpha=0.7, label="Lag 24 (diario)")
axes[1].axvline(x=168, color="#2ecc71", linestyle="--", alpha=0.7, label="Lag 168 (semanal)")
axes[1].legend(loc="upper right")

plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/01_acf_pacf.png", dpi=150, bbox_inches="tight")
plt.show()

### 3.7 Detección de gaps temporales

Se verifica la continuidad de la serie temporal identificando horas faltantes. Los gaps pueden deberse a interrupciones en la disponibilidad de la API, mantenimientos del sistema, o ausencia de reporte.

In [ ]:
# --- 6.7 Detección de gaps ---
full_range = pd.date_range(
    start=df["datetime"].min(),
    end=df["datetime"].max(),
    freq="h",
)
actual_timestamps = set(df["datetime"])
missing = sorted(set(full_range) - actual_timestamps)

print(f"Rango esperado: {full_range[0]} -> {full_range[-1]}")
print(f"Total horas esperadas: {len(full_range):,}")
print(f"Total horas presentes: {len(actual_timestamps):,}")
print(f"Horas faltantes: {len(missing):,}")

if missing:
    # Visualizar gaps
    gap_df = pd.DataFrame({"datetime": missing})
    gap_df["date"] = gap_df["datetime"].dt.date
    gap_counts = gap_df.groupby("date").size().reset_index(name="missing_hours")

    fig, ax = plt.subplots(figsize=(14, 4))
    ax.bar(
        pd.to_datetime(gap_counts["date"]),
        gap_counts["missing_hours"],
        color="#e74c3c",
        width=1,
    )
    ax.set_title("Horas Faltantes por Día (Gaps Temporales)", fontweight="bold")
    ax.set_xlabel("Fecha")
    ax.set_ylabel("Horas faltantes")
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    plt.tight_layout()
    plt.savefig(f"{RESULTS_DIR}/01_gaps_temporales.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("\nNo se detectaron gaps temporales. La serie está completa.")

### 3.8 Estadísticas descriptivas por período

Se calculan estadísticas descriptivas agrupadas por año, mes, día de la semana y hora para entender la variabilidad de la demanda en distintas granularidades temporales.

In [ ]:
# --- 6.8 Estadísticas descriptivas por período ---
df["year"] = df["datetime"].dt.year

# Por año
print("=" * 60)
print("ESTADÍSTICAS POR AÑO")
print("=" * 60)
yearly_stats = df.groupby("year")["demand_kwh"].agg(
    ["count", "mean", "std", "min", "median", "max"]
).round(1)
yearly_stats.columns = ["N", "Media", "Std", "Mín", "Mediana", "Máx"]
print(yearly_stats.to_string())

# Por mes (promedio de todos los años)
print(f"\n{'=' * 60}")
print("ESTADÍSTICAS POR MES (promedio multi-anual)")
print("=" * 60)
monthly_stats = df.groupby("month")["demand_kwh"].agg(
    ["mean", "std", "min", "max"]
).round(1)
monthly_stats.index = month_labels
monthly_stats.columns = ["Media", "Std", "Mín", "Máx"]
print(monthly_stats.to_string())

# Por hora
print(f"\n{'=' * 60}")
print("ESTADÍSTICAS POR HORA DEL DÍA")
print("=" * 60)
hourly_stats = df.groupby("hour")["demand_kwh"].agg(
    ["mean", "std", "min", "max"]
).round(1)
hourly_stats.index = [f"{h:02d}:00" for h in range(24)]
hourly_stats.columns = ["Media", "Std", "Mín", "Máx"]
print(hourly_stats.to_string())

# Visualizar tendencia anual
fig, ax = plt.subplots(figsize=(12, 5))
monthly_mean = df.groupby(["year", "month"])["demand_kwh"].mean().reset_index()
monthly_mean["date"] = pd.to_datetime(
    monthly_mean["year"].astype(str) + "-" + monthly_mean["month"].astype(str) + "-15"
)
ax.plot(monthly_mean["date"], monthly_mean["demand_kwh"], "o-",
        color="#2c3e50", markersize=4, linewidth=1.2)
ax.set_title("Demanda Promedio Mensual", fontweight="bold")
ax.set_xlabel("Fecha")
ax.set_ylabel("Demanda promedio (kWh)")
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_minor_locator(mdates.MonthLocator(bymonth=[4, 7, 10]))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/01_demanda_mensual.png", dpi=150, bbox_inches="tight")
plt.show()

### 3.9 Descomposición STL (Seasonal-Trend decomposition using LOESS)

Se aplica descomposición STL con dos periodicidades:
- **P=24:** Componente diario (patrón horario dentro de cada día).
- **P=168:** Componente semanal (patrón que se repite cada semana).

La descomposición STL separa la serie en tres componentes: tendencia, estacionalidad y residuo. Esto permite verificar visualmente la presencia de estacionalidad y tendencias antes de construir el modelo.

In [ ]:
# --- 6.9 Descomposición STL ---
# Preparar serie con índice temporal regular
ts = df.set_index("datetime")["demand_kwh"].copy()
ts = ts.asfreq("h")

# Imputar NaNs (si hay gaps) con interpolación lineal para que STL funcione
n_nans = ts.isna().sum()
if n_nans > 0:
    print(f"Imputando {n_nans} valores NaN con interpolación lineal para STL...")
    ts = ts.interpolate(method="linear")

# --- STL con periodo diario (P=24) ---
print("Ejecutando STL con período diario (P=24)...")
stl_daily = STL(ts, period=24, robust=True)
result_daily = stl_daily.fit()

fig, axes = plt.subplots(4, 1, figsize=(16, 12), sharex=True)
components = [
    ("Serie Original", ts, "#2c3e50"),
    ("Tendencia", result_daily.trend, "#e74c3c"),
    ("Estacionalidad Diaria (P=24)", result_daily.seasonal, "#3498db"),
    ("Residuo", result_daily.resid, "#95a5a6"),
]
for ax, (title, data, color) in zip(axes, components):
    ax.plot(data.index, data.values, linewidth=0.3, color=color, alpha=0.8)
    ax.set_ylabel("kWh")
    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))

axes[-1].set_xlabel("Fecha")
axes[-1].xaxis.set_major_locator(mdates.YearLocator())
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
fig.suptitle("Descomposición STL — Período Diario (P=24)", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/01_stl_diario.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
df_2022_jan = df[(df['datetime'].dt.year == 2022) & (df['datetime'].dt.month == 1)].copy()

# --- 6.9 Descomposición STL ---
# Preparar serie con índice temporal regular
ts_filtrado = df_2022_jan.set_index("datetime")["demand_kwh"].copy()
ts_filtrado = ts_filtrado.asfreq("h")

# Imputar NaNs (si hay gaps) con interpolación lineal para que STL funcione
n_nans = ts_filtrado.isna().sum()
if n_nans > 0:
    print(f"Imputando {n_nans} valores NaN con interpolación lineal para STL...")
    ts_filtrado = ts_filtrado.interpolate(method="linear")

# --- STL con periodo diario (P=24) ---
print("Ejecutando STL con período diario (P=24)...")
stl_daily = STL(ts_filtrado, period=24, robust=True)
result_daily = stl_daily.fit()

fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
components = [
    ("Serie Original", ts_filtrado, "#0052CC"),   # azul fuerte
    ("Tendencia", result_daily.trend, "#D7263D"), # rojo intenso
    ("Estacionalidad Diaria (P=24)", result_daily.seasonal, "#00A6D6"),
    ("Residuo", result_daily.resid, "#7B2CBF"),   # morado fuerte
]

for ax, (title, data, color) in zip(axes, components):
    ax.plot(data.index, data.values, linewidth=0.6, color=color, alpha=0.8)
    ax.set_ylabel("kWh")
    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))

axes[-1].set_xlabel("Fecha")
axes[-1].xaxis.set_major_locator(mdates.WeekdayLocator()) # Corrected from DaysLocator()
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%D"))
fig.suptitle("Descomposición STL — Período Diario (P=24)", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/01_stl_diario_mes_filtrado.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# --- STL con periodo semanal (P=168) ---
print("Ejecutando STL con período semanal (P=168)...")
stl_weekly = STL(ts, period=168, robust=True)
result_weekly = stl_weekly.fit()

fig, axes = plt.subplots(4, 1, figsize=(16, 12), sharex=True)
components = [
    ("Serie Original", ts, "#2c3e50"),
    ("Tendencia", result_weekly.trend, "#e74c3c"),
    ("Estacionalidad Semanal (P=168)", result_weekly.seasonal, "#27ae60"),
    ("Residuo", result_weekly.resid, "#95a5a6"),
]
for ax, (title, data, color) in zip(axes, components):
    ax.plot(data.index, data.values, linewidth=0.3, color=color, alpha=0.8)
    ax.set_ylabel("kWh")
    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))

axes[-1].set_xlabel("Fecha")
axes[-1].xaxis.set_major_locator(mdates.YearLocator())
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
fig.suptitle("Descomposición STL — Período Semanal (P=168)", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/01_stl_semanal.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# --- STL con periodo semanal (P=168) ---
print("Ejecutando STL con período semanal (P=168)...")
stl_weekly = STL(ts_filtrado, period=168, robust=True)
result_weekly = stl_weekly.fit()

fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
components = [
    ("Serie Original", ts_filtrado, "#0052CC"),   # azul fuerte
    ("Tendencia", result_weekly.trend, "#D7263D"), # rojo intenso
    ("Estacionalidad Semanal (P=168)", result_weekly.seasonal, "#00A6D6"),
    ("Residuo", result_weekly.resid, "#7B2CBF"),   # morado fuerte
]

for ax, (title, data, color) in zip(axes, components):
    ax.plot(data.index, data.values, linewidth=0.6, color=color, alpha=0.8)
    ax.set_ylabel("kWh")
    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))

axes[-1].set_xlabel("Fecha")
axes[-1].xaxis.set_major_locator(mdates.WeekdayLocator()) # Corrected from DaysLocator()
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%D"))
fig.suptitle("Descomposición STL — Período Semanal (P=168)", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/01_stl_semanal_mes_filtrado.png", dpi=150, bbox_inches="tight")
plt.show()

---

## 4. Analisis de estacioinariedad

In [ ]:
from statsmodels.tsa.stattools import adfuller, kpss

def run_stationarity_tests(series, series_name):
    """Runs ADF and KPSS tests on a given series and returns formatted results."""
    results = []

    # --- ADF Test ---
    adf_result = adfuller(series, autolag='AIC')
    adf_statistic = adf_result[0]
    adf_pvalue = adf_result[1]
    adf_critical_value = adf_result[4]['5%']
    adf_conclusion = "Estacionaria (rechaza H0)" if adf_pvalue <= 0.05 and adf_statistic < adf_critical_value else "No estacionaria (no rechaza H0)"
    results.append({
        "Serie": series_name,
        "Prueba": "ADF (Augmented Dickey-Fuller)",
        "Estadístico": round(adf_statistic, 4),
        "p-valor": round(adf_pvalue, 4),
        "Valor crítico (5%)": adf_critical_value,
        "Conclusión": adf_conclusion,
    })

    # --- KPSS Test ---
    kpss_result = kpss(series, regression='c', nlags='auto') # 'c' for level stationarity
    kpss_statistic = kpss_result[0]
    kpss_pvalue = kpss_result[1]
    kpss_critical_value = kpss_result[3]['5%']
    kpss_conclusion = "No estacionaria (rechaza H0)" if kpss_pvalue <= 0.05 and kpss_statistic > kpss_critical_value else "Estacionaria (no rechaza H0)"
    results.append({
        "Serie": series_name,
        "Prueba": "KPSS (Kwiatkowski-Phillips-Schmidt-Shin)",
        "Estadístico": round(kpss_statistic, 4),
        "p-valor": round(kpss_pvalue, 4),
        "Valor crítico (5%)": kpss_critical_value,
        "Conclusión": kpss_conclusion,
    })

    return results

print("=" * 60)
print("ANÁLISIS DE ESTACIONARIEDAD")
print("=" * 60)

# --- Preparar serie original para el análisis ---
# 'ts' variable should already be defined from previous STL decomposition cells
# If not, recreate it:
if 'ts' not in locals():
    ts = df.set_index("datetime")["demand_kwh"].copy()
    ts = ts.asfreq("h")
    if ts.isna().sum() > 0:
        ts = ts.interpolate(method="linear")

# --- Preparar residuos STL (usando daily STL decomposition results) ---
# 'result_daily' should be available from previous STL decomposition cells
# If not, run daily STL decomposition to get the residuals
if 'result_daily' not in locals():
    print("[WARN] result_daily not found. Running daily STL for residuals...")
    stl_daily = STL(ts, period=24, robust=True)
    result_daily = stl_daily.fit()

# Run tests
stationarity_results = []
stationarity_results.extend(run_stationarity_tests(ts, "Serie Original"))
stationarity_results.extend(run_stationarity_tests(result_daily.resid.dropna(), "Residuos STL (Diario)"))

# Display results in a DataFrame
stationarity_df = pd.DataFrame(stationarity_results)
print(stationarity_df.to_string(index=False))


In [ ]:
stationarity_df.to_csv(f"{RESULTS_DIR}/stationarity_results.csv", index=False)

---

## 5. Guardado de datos

Se guardan los datos descargados en formato **Parquet** (eficiente en tamaño y velocidad de lectura) junto con un archivo de **metadatos JSON** que documenta las características del dataset descargado.

In [ ]:
# --- 7. Guardar datos ---

# Preparar DataFrame limpio para guardar
df_save = df[["datetime", "demand_kwh"]].copy()
df_save = df_save.sort_values("datetime").reset_index(drop=True)

# Guardar Parquet
parquet_path = f"{DATA_DIR}/raw/demand.parquet"
df_save.to_parquet(parquet_path, engine="pyarrow", index=False)
parquet_size_mb = os.path.getsize(parquet_path) / (1024 * 1024)
print(f"Parquet guardado: {parquet_path} ({parquet_size_mb:.2f} MB)")

# Construir metadatos
metadata = {
    "project": "MoE Electricity Demand Forecasting - SIN Colombia",
    "notebook": "01_data_acquisition",
    "variable": "DemandaReal (DemaSIN)",
    "source": "XM S.A. E.S.P. - API SINERGOX",
    "api_url": API_URL,
    "resolution": "hourly",
    "unit": "kWh",
    "start_date": str(df_save["datetime"].min()),
    "end_date": str(df_save["datetime"].max()),
    "n_records": len(df_save),
    "n_days": int(df_save["datetime"].dt.date.nunique()),
    "n_nulls": int(df_save["demand_kwh"].isna().sum()),
    "demand_stats": {
        "mean_kwh": round(float(df_save["demand_kwh"].mean()), 2),
        "std_kwh": round(float(df_save["demand_kwh"].std()), 2),
        "min_kwh": round(float(df_save["demand_kwh"].min()), 2),
        "max_kwh": round(float(df_save["demand_kwh"].max()), 2),
    },
    "detected_gaps": validation_report.get("gap_blocks", []),
    "n_missing_hours": validation_report.get("n_missing_hours", 0),
    "download_timestamp": DOWNLOAD_TIMESTAMP,
    "parquet_file": "demand.parquet",
    "parquet_size_mb": round(parquet_size_mb, 2),
}

# Guardar metadatos JSON
metadata_path = f"{DATA_DIR}/raw/metadata.json"
with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False, default=str)

print(f"Metadatos guardados: {metadata_path}")
print(f"\nResumen:")
print(f"  Registros: {metadata['n_records']:,}")
print(f"  Días: {metadata['n_days']:,}")
print(f"  Período: {metadata['start_date']} -> {metadata['end_date']}")
print(f"  Gaps detectados: {metadata['n_missing_hours']} horas")
print(f"  Tamaño Parquet: {metadata['parquet_size_mb']} MB")